# 13 - Delta features on the company-month panel

Notebook 12 left me with a 33-month company-month panel: 49.6M rows over ~2.04M companies, verified to reproduce notebook 1 exactly at the June 2026 vintage. That panel still only describes companies *as they were* in each month, though. What I actually want to model is **change**, because change is the buying signal: a company whose charges jumped three in six months is behaving differently from one that has sat still since 2023, even if both look identical today.

So this notebook computes the delta features. Every one of them is **strictly backward-looking** from month `t`: nothing here is allowed to peek at `t+1` or later, because these become model inputs and the labels come from the future. Getting that boundary wrong is the classic way to build a model with a beautiful AUC that is worthless in production.

The logic lives in `src/features/panel.py` (`DELTA_SQL` / `build_deltas`) so it is reproducible and testable; I drive it from here and then check it.

## The June 2025 hole, and why it dictates the whole design

This is the single most important thing in this notebook, so I want to write it down properly rather than bury it in a comment. It also took me three attempts to get right, so the dead ends are worth recording.

June 2025 does not exist on the Companies House server. It is a real hole, not a download failure. My panel therefore has 33 months but spans 34 calendar months.

The naive way to compute a 3-month delta is a positional `LAG(3)`: go back three *rows*. On a panel with a hole that is silently wrong. Standing in September 2025 and stepping back three rows lands on **May** 2025, because June has no row. So `d_charges_3m` would quietly become a four-month delta, and nothing would error, warn, or look odd. I would just have a band of subtly corrupted features running through the middle of my training data.

**What I tried first.** Self-joining each row to its lag on an explicit `m - INTERVAL 3 MONTH`. Semantically perfect, but it builds four hash tables over 49.6M rows; it spilled 27 GB without finishing on this machine.

**What I tried second.** Keep the cheap positional `LAG(3)`, but also grab the lag row's own month and reject the value unless it is exactly three calendar months back. Never wrong, but **badly lossy**, and I only caught this because I inspected the gap region instead of trusting the asserts. Sitting in July 2025, the true `t-3` (April 2025) exists and is perfectly usable, but it sits at `LAG(2)` because June is absent, so the check rejects it. Worse, after the hole `LAG(12)` is off by one for a whole year, which would have made `d_charges_12m` NULL from July 2025 to May 2026 and gutted the 12-month growth label.

**What I settled on: a dense month spine.** I build every company against all 34 calendar months (`universe CROSS JOIN months`, then `LEFT JOIN` the panel), so June 2025 exists as an explicit all-NULL row. Positional `LAG(N)` is now *exactly* N calendar months back by construction. Reaching over the hole lands on the NULL row and the delta is NULL; months either side compute normally. One join and one sort, and it handles per-company absence (not yet incorporated, or gone from the register) for free rather than as a special case.

A `present` flag distinguishes "no row for this company that month" from a genuine NULL value. That matters because `IS DISTINCT FROM` never returns NULL, so without the guard an absent lag month would read as "changed" for things like `sic_changed_12m`.

For streaks and recency I use window functions over the observed months and express the answer with `datediff('month', ...)`, so the output stays in real calendar months. The gap month must not reset everyone's streak, so run-breaks are only evaluated on `present` rows and compare against the previous *observed* month via `LAG(... IGNORE NULLS)`.

The trailing-12-month counts use a `RANGE ... INTERVAL 11 MONTHS PRECEDING` frame, keyed on the date rather than the row, so it is calendar-aware for free: the missing month contributes nothing rather than dragging an extra month of history into the window.

## Setup

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

import duckdb
import pandas as pd

from src.features import panel

con = duckdb.connect()
con.execute("PRAGMA disable_progress_bar")
con.execute("PRAGMA threads=22")

PANEL_GLOB = (REPO_ROOT / panel.PANEL_DIR / "**" / "*.parquet").as_posix()
DELTA_GLOB = (REPO_ROOT / panel.DELTA_DIR / "**" / "*.parquet").as_posix()

print(f"{len(panel.DELTA_FEATURE_COLS)} delta features defined")
for c in panel.DELTA_FEATURE_COLS:
    print(" ", c)

25 delta features defined
  d_charges_3m
  d_charges_6m
  d_charges_12m
  d_outstanding_3m
  d_outstanding_6m
  d_outstanding_12m
  d_satisfied_12m
  debt_ratio_trend_12m
  new_charge_events_12m
  months_since_last_new_charge
  status_changed
  months_in_current_status
  ever_distressed_before
  segment_upgraded_12m
  segment_downgraded_12m
  months_since_segment_change
  accounts_overdue_streak_months
  accounts_stale_streak_months
  confstmt_late
  days_to_next_accounts_due
  months_since_last_accounts_filing
  months_since_last_confstmt
  sic_changed_12m
  name_changed_12m
  postcode_changed_12m


## What each delta means

Grouped by the story they tell:

**Charge dynamics** (the lending signal). `d_charges_3m/6m/12m` and `d_outstanding_3m/6m/12m` are the change in number of mortgage charges and outstanding charges. `d_satisfied_12m` counts charges paid off. `new_charge_events_12m` counts how many months in the trailing year saw a *new* charge appear, which separates one big financing event from a company that keeps borrowing. `months_since_last_new_charge` is recency. `debt_ratio_trend_12m` is the change in outstanding/total.

**Status** (the distress signal). `status_changed` flags a change since last month. `months_in_current_status` is how long it has been sitting where it is. `ever_distressed_before` marks a company that has been non-Active at some point strictly *before* `t`, which is a strong prior on it happening again.

**Size** (the growth signal). `segment_upgraded_12m` / `segment_downgraded_12m` compare size tier against 12 months ago, and `months_since_segment_change` is recency.

**Filing behaviour** (an early warning that tends to lead the formal status). `accounts_overdue_streak_months` and `accounts_stale_streak_months` are how long accounts have been late / stale, both built with the same run-length logic. `confstmt_late` and `days_to_next_accounts_due` are the current deadline position. And `months_since_last_accounts_filing` / `months_since_last_confstmt` are the "did they stop filing" signal: they cycle roughly annually while a company keeps filing and only ever climb once it stops, which usually leads the status flipping to strike-off or insolvency.

**Identity drift.** `sic_changed_12m`, `name_changed_12m`, `postcode_changed_12m` (a pivot, rebrand or relocation).

Deliberately absent: **1-month diffs**. On a register that updates this slowly they are ~99% zeros and carry almost no signal, so they are noise with a column name.

## Run the build

Unlike the panel build, this cannot be a per-month loop: a 12-month lag reaches across partitions, so the whole history has to be in scope for one query. It sets a DuckDB memory limit and a temp directory so it spills to disk rather than trying to hold the spine in RAM. Output is written back out partitioned by `snapshot_date`, matching the panel layout.

On my machine this runs in about 5 minutes, peaks around 9 GB resident and spills roughly 12 GB, so it wants some free disk. One thing worth watching: the row count out must exactly equal the row count in (**49,556,152**). The spine deliberately invents rows for the gap month and for months where a company is not on the register, and the final `WHERE present` has to drop every single one of them again. If this number drifts, the spine is leaking synthetic rows into the panel.

In [2]:
n_rows = panel.build_deltas(threads=22)
print(f"delta rows: {n_rows:,}")

# The spine must not leak synthetic rows: one delta row per panel row, exactly.
n_panel = con.execute(f"SELECT count(*) FROM read_parquet('{PANEL_GLOB}')").fetchone()[0]
print(f"panel rows: {n_panel:,}")
assert n_rows == n_panel, "spine leaked synthetic rows into the delta table"
print("PASS: every spine row with no underlying panel row was dropped.")

delta rows: 49,556,152
panel rows: 49,556,152
PASS: every spine row with no underlying panel row was dropped.


## Verification - the gap is handled, not papered over

This is the check the whole design exists for, so it gets tested directly rather than assumed.

Because June 2025 is missing, I can predict exactly which months must have a NULL 3-month delta: any month whose `t-3` lands on June 2025, i.e. **September 2025**. Likewise the 12-month delta must be NULL for **June 2026**, whose `t-12` is June 2025.

And critically, the months either side must be **fine**: August 2025 reaches back to May 2025, which exists, so it must *not* be NULL. That is the pair that would break under a positional `LAG(3)`, which is why I check both directions rather than just "is it null".

In [3]:
# Share of rows with a non-null 3m / 12m delta, per month.
gap = con.execute(f"""
    SELECT snapshot_date,
           round(100.0 * count(d_charges_3m)  / count(*), 1) AS pct_3m_present,
           round(100.0 * count(d_charges_12m) / count(*), 1) AS pct_12m_present
    FROM read_parquet('{DELTA_GLOB}')
    GROUP BY snapshot_date ORDER BY snapshot_date
""").df()
print(gap.to_string(index=False))

snapshot_date  pct_3m_present  pct_12m_present
   2023-10-01             0.0              0.0
   2023-11-01             0.0              0.0
   2023-12-01             0.0              0.0
   2024-01-01            96.5              0.0
   2024-02-01            96.0              0.0
   2024-03-01            96.3              0.0
   2024-04-01            96.0              0.0
   2024-05-01            96.0              0.0
   2024-06-01            96.1              0.0
   2024-07-01            96.5              0.0
   2024-08-01            97.0              0.0
   2024-09-01            96.9              0.0
   2024-10-01            96.9             86.2
   2024-11-01            96.8             86.2
   2024-12-01            96.8             86.5
   2025-01-01            97.0             86.6
   2025-02-01            97.0             87.1
   2025-03-01            96.9             87.1
   2025-04-01            96.5             87.2
   2025-05-01            96.5             87.5
   2025-07-01

In [4]:
g = gap.set_index(gap["snapshot_date"].astype(str))

# t-3 == 2025-06 (missing) -> must be 0% present
assert g.loc["2025-09-01", "pct_3m_present"] == 0.0, "Sep 2025 3m delta should be entirely NULL"
# t-12 == 2025-06 (missing) -> must be 0% present
assert g.loc["2026-06-01", "pct_12m_present"] == 0.0, "Jun 2026 12m delta should be entirely NULL"
# The neighbours reach across the hole to a month that exists, so they must survive.
assert g.loc["2025-08-01", "pct_3m_present"] > 90, "Aug 2025 (t-3 = May 2025) should be present"
assert g.loc["2025-10-01", "pct_3m_present"] > 90, "Oct 2025 (t-3 = Jul 2025) should be present"

print("Gap handled correctly:")
print("  Sep 2025 3m  ->", g.loc["2025-09-01", "pct_3m_present"], "% present (NULL, as it must be)")
print("  Aug 2025 3m  ->", g.loc["2025-08-01", "pct_3m_present"], "% present (reaches May 2025, fine)")
print("  Jun 2026 12m ->", g.loc["2026-06-01", "pct_12m_present"], "% present (NULL, as it must be)")
print("\nA positional LAG(3) would have silently returned a 4-month delta for Sep 2025 instead.")

Gap handled correctly:
  Sep 2025 3m  -> 0.0 % present (NULL, as it must be)
  Aug 2025 3m  -> 96.4 % present (reaches May 2025, fine)
  Jun 2026 12m -> 0.0 % present (NULL, as it must be)

A positional LAG(3) would have silently returned a 4-month delta for Sep 2025 instead.


The first months of the panel also show NULLs, which is correct and expected: in October 2023 there is simply no history behind the panel to difference against. The 3-month deltas start in January 2024 and the 12-month deltas in October 2024. That is a real constraint on how many origin months I get for the 12-month growth label, and it is why I extended the history to the full 33 months rather than the 24 I first planned.

## Verification - no future leakage

The deltas are model inputs and the labels come from the future, so I want positive evidence that nothing here reaches forward. Two checks:

1. `ever_distressed_before` must be **strictly** before `t`. A company that is non-Active for the first time in month `t` must still have `ever_distressed_before = false` in that month; if it were true, the feature would be labelling itself and any distress model would score suspiciously well for the wrong reason.
2. Recomputing a delta for one month using only data at or before `t` must reproduce the stored value.

In [5]:
# A company's FIRST non-Active month must not be flagged as previously distressed.
leak = con.execute(f"""
    WITH d AS (
        SELECT "CompanyNumber", snapshot_date, is_active, ever_distressed_before,
               MIN(CASE WHEN NOT is_active THEN snapshot_date END)
                   OVER (PARTITION BY "CompanyNumber") AS first_bad
        FROM read_parquet('{DELTA_GLOB}')
    )
    SELECT count(*) AS violations
    FROM d
    WHERE snapshot_date = first_bad AND ever_distressed_before
""").fetchone()[0]
print("rows where the first distress month claims prior distress:", leak)
assert leak == 0, "ever_distressed_before is leaking the current month"
print("PASS: ever_distressed_before looks strictly backwards.")

rows where the first distress month claims prior distress: 0
PASS: ever_distressed_before looks strictly backwards.


In [6]:
# Independent recomputation: d_charges_3m at Aug 2025 must equal charges(Aug) - charges(May),
# May being the correct calendar month 3 back, given June is missing.
check = con.execute(f"""
    SELECT count(*) AS mismatches
    FROM (
        SELECT a."CompanyNumber",
               a.d_charges_3m AS stored,
               a."Mortgages.NumMortCharges" - b."Mortgages.NumMortCharges" AS recomputed
        FROM read_parquet('{DELTA_GLOB}') a
        JOIN read_parquet('{PANEL_GLOB}') b
          ON b."CompanyNumber" = a."CompanyNumber"
         AND b.snapshot_date = DATE '2025-05-01'
        WHERE a.snapshot_date = DATE '2025-08-01'
    )
    WHERE stored IS DISTINCT FROM recomputed
""").fetchone()[0]
print("mismatches vs independent recomputation:", check)
assert check == 0
print("PASS: the 3-month delta really is a 3-calendar-month delta.")

mismatches vs independent recomputation: 0
PASS: the 3-month delta really is a 3-calendar-month delta.


## Do the deltas actually carry signal?

A feature that is 99.9% zeros is a column, not a signal. Before I build anything on top of these I want to know how often they actually move, and whether the movement is economically sensible. This is also a sanity check on the plan's decision to drop 1-month diffs.

In [7]:
# How often is each delta non-zero / true? (latest month with full 12m history behind it)
move = con.execute(f"""
    SELECT
        count(*) AS n,
        round(100.0 * avg(CASE WHEN d_charges_3m  <> 0 THEN 1 ELSE 0 END), 3) AS pct_d_charges_3m,
        round(100.0 * avg(CASE WHEN d_charges_12m <> 0 THEN 1 ELSE 0 END), 3) AS pct_d_charges_12m,
        round(100.0 * avg(CASE WHEN new_charge_events_12m > 0 THEN 1 ELSE 0 END), 3) AS pct_any_new_charge_12m,
        round(100.0 * avg(CASE WHEN segment_upgraded_12m THEN 1 ELSE 0 END), 3) AS pct_upgraded_12m,
        round(100.0 * avg(CASE WHEN segment_downgraded_12m THEN 1 ELSE 0 END), 3) AS pct_downgraded_12m,
        round(100.0 * avg(CASE WHEN accounts_overdue THEN 1 ELSE 0 END), 3) AS pct_overdue,
        round(100.0 * avg(CASE WHEN ever_distressed_before THEN 1 ELSE 0 END), 3) AS pct_ever_distressed
    FROM read_parquet('{DELTA_GLOB}')
    WHERE snapshot_date = DATE '2026-07-01'
""").df()
print(move.T.to_string(header=False))

n                       1531094.000
pct_d_charges_3m              0.260
pct_d_charges_12m             0.882
pct_any_new_charge_12m        0.924
pct_upgraded_12m              1.241
pct_downgraded_12m            1.442
pct_overdue                   6.513
pct_ever_distressed          10.899


In [8]:
# Does charge activity line up with distress the way I'd expect?
# (This is a correlation on the panel, not a causal claim; I just want to see the
#  features separate the groups before I trust a model built on them.)
sep = con.execute(f"""
    SELECT
        CASE WHEN is_active THEN 'Active' ELSE 'Not active' END AS grp,
        count(*) AS n,
        round(avg(new_charge_events_12m), 3) AS avg_new_charge_events_12m,
        round(avg(CASE WHEN accounts_overdue THEN 1.0 ELSE 0.0 END), 3) AS rate_overdue,
        round(avg(accounts_overdue_streak_months), 2) AS avg_overdue_streak,
        round(avg(company_age_years), 1) AS avg_age
    FROM read_parquet('{DELTA_GLOB}')
    WHERE snapshot_date = DATE '2026-07-01'
    GROUP BY 1
""").df()
print(sep.to_string(index=False))

       grp       n  avg_new_charge_events_12m  rate_overdue  avg_overdue_streak  avg_age
    Active 1409284                      0.011         0.018                0.09      9.4
Not active  121810                      0.002         0.611               13.13      9.6


## Where this leaves me

The panel now has ~25 backward-looking delta features on top of the static ones, the June 2025 hole is provably handled (NULL where it must be, intact either side), and `ever_distressed_before` is provably strictly backward-looking.

Still to come before any modelling:

- **Contracts Finder**, joined *as-of* each month. Notebook 5 already pulls a real Companies House number from the OCDS `parties` block, so this is a clean join on `CompanyNumber` rather than fuzzy name matching. It has to be as-of, though: a static join would stamp a 2026 contract win onto a 2024 row and leak the future into the features.
- **The three targets** (lending 3m, distress 6m, growth 12m), which are the mirror image of everything in this notebook: features come from `t`, labels come strictly from `t+1` onwards.